<a href="https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 – ML Task Framing

## My Lane
**Lane:** Refresh / Content Opportunity Scoring

The goal is to help a content team decide which pages deserve limited reviewer attention first.

## 1. My Lane as an ML Task

**Task Type:** Ranking / Scoring

The practical problem is to rank content pages by review priority so the SEO/content team can decide which pages should be investigated first for a possible content opportunity.

A page may deserve attention for different reasons, such as weak click-through performance despite strong visibility, declining performance, age, or lack of recent updates.

The model is therefore not intended to automatically decide that a page must be refreshed. Instead, it should produce a ranked review queue that helps a human reviewer decide where to spend limited time first.

## 2. Target (or Proxy)

Ideally, the model would learn from a future observed outcome showing whether reviewing or refreshing a page led to a useful improvement.

The starter dataset does not contain a direct label such as `needs_refresh` or a measured post-refresh outcome. Therefore, a development-time proxy is needed.

One available signal is `trend_direction`, which describes whether recent performance is moving up, down, or remaining stable.

For development and evaluation, I can use a page with `trend_direction = "down"` as a proxy indicator of a page that may deserve attention.

I will therefore create a temporary binary proxy called `proxy_decline`:

- 1 = observed trend is down
- 0 = observed trend is not down

This proxy should not be interpreted as ground truth for whether a page actually needs a refresh. A downward trend can have many causes, and a page with a stable or upward trend may still have another content opportunity.

In [ ]:
import pandas as pd

# Load the anonymized FlyRank dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create a development-time proxy from trend direction
df["proxy_decline"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("\nProxy distribution:")
print(df["proxy_decline"].value_counts())

df.head()

## 3. Success Metric

The main evaluation metric is **Precision@K**, with particular attention to Precision@20.

The content team has limited review capacity, so the most important part of the ranking is the top of the queue.

Precision@20 measures how many of the 20 highest-ranked pages match the selected development-time proxy.

For example, if 18 of the top 20 ranked pages have the proxy outcome of 1:

**Precision@20 = 18 / 20 = 0.90**

This metric is useful because it reflects the practical question of whether the highest-priority recommendations contain a large proportion of pages matching the current proxy outcome.

The proxy limitation will be stated clearly: a high Precision@20 against this proxy does not prove that the recommended pages will benefit from a future content refresh.

## 4. Unit of Analysis

**One row = one anonymized content item.**

Each row contains observable signals about the content item, including search visibility, click-through performance, search position, content age, and freshness.

The eventual model should produce one opportunity score for each content item. Sorting these scores creates the ranked review queue.

In [ ]:
print("Dataset shape:", df.shape)

candidate_columns = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "proxy_decline"
]

available_columns = [
    col for col in candidate_columns
    if col in df.columns
]

df[available_columns].head()

## 5. Why ML May Beat a Fixed Rule

A simple rule-based approach can provide a useful starting point. For example, a rule might prioritize pages with high impressions, a relatively strong search position, and low CTR.

The advantage of a simple rule is that it is transparent and easy for a reviewer to understand. However, a fixed rule may not capture more complicated relationships between several signals.

For example, two pages can have similar impressions but very different content ages, positions, CTRs, or update histories. These combinations may provide different levels of evidence for a content opportunity.

A machine learning model can consider several signals together and learn a scoring function from the available development data.

However, ML is not automatically better. The model must be compared with the simple Week-4 baseline using the same data, split, and evaluation metric. If the more complex model does not provide useful improvement, the simpler baseline may be the better choice.

## 6. Real Business Action

The ranked output is intended to support a human content-review workflow.

The content team can use the ranking to:

- Prioritize which pages to inspect first.
- Investigate pages with stronger evidence of a potential opportunity.
- Decide whether a page should be refreshed, expanded, protected, pruned, or simply monitored.
- Spend limited review time more efficiently.

The model does not make the final editorial decision. It provides decision support and prioritization.

## 7. Leakage and Data Boundaries

The model should only use information that would be available when the review-priority decision is made.

If `trend_direction` is used to create the development-time proxy, it should not also be used as a predictive feature for that same proxy because that would directly expose the target to the model.

Similarly, future performance measures or post-intervention outcomes should not be used as features when they would only become known after the decision.

The Week-3 data contract will define the final feature set and exclusions in more detail.

## 8. Final ML Task Statement

**Problem:** Prioritize content items for limited human review.

**Task type:** Ranking / scoring.

**Unit:** One anonymized content item per row.

**Output:** A continuous opportunity score used to create a ranked review queue.

**Development-time proxy:** `trend_direction = "down"`, represented as `proxy_decline`.

**Primary metric:** Precision@20, with Precision@50 as an additional diagnostic.

**Baseline:** The transparent rule-based content opportunity score developed in Week 4.

**Modeling principle:** Prefer a learned model only when it provides measurable improvement over the simple baseline without leakage or unnecessary complexity.

**Human role:** The model recommends review priority; a human reviewer makes the final content decision.

## 9. Self-Check

By completing this notebook, I can now explain:

- Why the problem is a ranking/scoring task.
- Why the goal is content-opportunity prioritization rather than automatic refresh decisions.
- What the development-time proxy is and why it is only a proxy.
- Why Precision@20 is useful for a limited review queue.
- What one row in the dataset represents.
- Why a simple baseline should be established before adding model complexity.
- Why the ML model must be compared with the Week-4 baseline on the same evaluation setup.
- Why leakage must be controlled when defining features and outcomes.
- How the ranked output supports a human content-review decision.